# Stage 2 Asset Generator — Style-Conditioned (SDXL + IP-Adapter)

Generates character and place images that match a specific reference style image, using **IP-Adapter** for style conditioning instead of relying on text description alone.

**Why this instead of the FLUX version:** text prompts can't reliably pin down a specific painterly/watercolor style. IP-Adapter feeds your actual reference image into the model as a style anchor, which is far more consistent.

**Model:** Stable Diffusion XL base 1.0 (CreativeML Open RAIL++-M — commercial use permitted) + IP-Adapter (`h94/IP-Adapter`, Apache 2.0).

**Runtime:** Runtime -> Change runtime type -> GPU (T4 is fine — SDXL is much lighter than FLUX).

## 1. Install dependencies

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

## 2. Mount Google Drive

Keeps your style reference and generated assets persistent across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_ROOT = '/content/drive/MyDrive/panchatantra_assets'
import os
os.makedirs(OUTPUT_ROOT, exist_ok=True)

## 3. Load your style reference image

Run the upload cell once and pick your reference image (the one you want every character/place to visually match). It gets saved to Drive so you don't need to re-upload next session — after the first run, the loader cell below will find it automatically.

In [ ]:
from pathlib import Path
from PIL import Image

STYLE_REF_PATH = Path(OUTPUT_ROOT) / 'style_reference.png'

if not STYLE_REF_PATH.exists():
    from google.colab import files
    print("No saved style reference found — please upload one now.")
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    Image.open(uploaded_name).convert("RGB").save(STYLE_REF_PATH)
    print(f"Saved to {STYLE_REF_PATH} for future sessions.")
else:
    print(f"Using existing style reference: {STYLE_REF_PATH}")

style_reference = Image.open(STYLE_REF_PATH).convert("RGB")
style_reference

## 4. Load SDXL + IP-Adapter

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16
).to("cuda")

pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="sdxl_models",
    weight_name="ip-adapter-plus_sdxl_vit-h.safetensors"
)

IP_ADAPTER_SCALE = 0.6  # 0.4-0.5 = prompt obeys more, 0.7-0.8 = style match stronger
pipe.set_ip_adapter_scale(IP_ADAPTER_SCALE)

## 5. Sample generation — test before running the full batch

Generate one image to check the style match and tune `IP_ADAPTER_SCALE` before spending time on the full character/place batch. Re-run this cell after changing the scale in the cell above (call `pipe.set_ip_adapter_scale(...)` again) to compare.

In [ ]:
sample_prompt = (
    "a friendly young brown monkey with big expressive round eyes, wearing a "
    "small red vest, long curled tail, cheerful open-mouth smile, standing pose, "
    "full body, forest background"
)
sample_negative_prompt = (
    "photorealistic, 3d render, flat vector, watermark, text, signature, "
    "blurry, extra limbs, deformed, low quality"
)

sample_image = pipe(
    prompt=sample_prompt,
    negative_prompt=sample_negative_prompt,
    ip_adapter_image=style_reference,
    num_inference_steps=30,
    guidance_scale=6.0,
    generator=torch.Generator("cuda").manual_seed(1001),
).images[0]

sample_image

**Check the result above.** If the style doesn't match closely enough, raise `IP_ADAPTER_SCALE` (e.g. 0.7-0.8) in cell 4 and re-run cells 4-5. If it's ignoring your character description too much (style is right but content is wrong), lower it (e.g. 0.4-0.5). Once you're happy, move on to the full batch below.

## 6. Story config

Content descriptions only — the visual style now comes from the reference image via IP-Adapter, so `art_style` here just holds a light-touch style hint plus the negative prompt.

In [ ]:
config = {
  "story_id": "monkey_and_crocodile",
  "style": {
    "art_style": "detailed storybook illustration, forest setting",
    "negative_prompt": "photorealistic, 3d render, flat vector, watermark, text, signature, blurry, extra limbs, deformed, low quality",
    "character_size": [1024, 1024],
    "place_size": [1344, 768]
  },
  "characters": [
    {
      "id": "monkey",
      "name": "Chintu the Monkey",
      "description": "a friendly young brown monkey with big expressive round eyes, wearing a small red vest, long curled tail, cheerful open-mouth smile, standing pose, full body, plain neutral background"
    },
    {
      "id": "crocodile",
      "name": "Kalu the Crocodile",
      "description": "a chubby green crocodile with rounded snout, small friendly eyes, textured back scales stylized as soft bumps, short stubby legs, standing pose, full body, plain neutral background"
    }
  ],
  "places": [
    {
      "id": "riverbank",
      "name": "River Bank",
      "description": "a lush green riverbank at soft morning light, a large fruit tree with overhanging branches on one side, calm blue river with gentle ripples, distant hills, wide establishing shot, no characters"
    },
    {
      "id": "river_middle",
      "name": "Middle of the River",
      "description": "view from the surface of a calm wide river, gentle ripples, soft morning sky reflected in the water, a few lily pads, wide shot, no characters"
    }
  ]
}

VARIANTS_PER_SUBJECT = 4

## 7. Generation function (batch, style-conditioned)

In [ ]:
import json

def build_prompt(subject_description, style):
    return f"{subject_description}, {style['art_style']}"

def generate_variants(subject_id, subject_description, style, size, out_dir, num_variants, manifest, category):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    prompt = build_prompt(subject_description, style)

    manifest_entry = {"prompt": prompt, "ip_adapter_scale": IP_ADAPTER_SCALE, "variants": []}

    for i in range(1, num_variants + 1):
        print(f"  [{category}:{subject_id}] generating variant {i}/{num_variants}...")
        image = pipe(
            prompt=prompt,
            negative_prompt=style.get("negative_prompt", ""),
            ip_adapter_image=style_reference,
            width=size[0],
            height=size[1],
            num_inference_steps=30,
            guidance_scale=6.0,
            generator=torch.Generator("cuda").manual_seed(1000 + i),
        ).images[0]

        variant_path = out_dir / f"variant_{i}.png"
        image.save(variant_path)
        manifest_entry["variants"].append(str(variant_path))
        torch.cuda.empty_cache()

    manifest[category][subject_id] = manifest_entry

## 8. Run generation for all characters and places

In [ ]:
story_id = config["story_id"]
style = config["style"]
base_dir = Path(OUTPUT_ROOT) / story_id

manifest = {"story_id": story_id, "style": style, "characters": {}, "places": {}}

for char in config.get("characters", []):
    generate_variants(
        subject_id=char["id"],
        subject_description=char["description"],
        style=style,
        size=style["character_size"],
        out_dir=base_dir / "characters" / char["id"],
        num_variants=VARIANTS_PER_SUBJECT,
        manifest=manifest,
        category="characters",
    )

for place in config.get("places", []):
    generate_variants(
        subject_id=place["id"],
        subject_description=place["description"],
        style=style,
        size=style["place_size"],
        out_dir=base_dir / "places" / place["id"],
        num_variants=VARIANTS_PER_SUBJECT,
        manifest=manifest,
        category="places",
    )

manifest_path = base_dir / "manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Done. Manifest written to {manifest_path}")

## 9. Review variants and pick canonical images

In [ ]:
import matplotlib.pyplot as plt

def show_variants(category, subject_id):
    entry = manifest[category][subject_id]
    fig, axes = plt.subplots(1, len(entry["variants"]), figsize=(4 * len(entry["variants"]), 4))
    for ax, path in zip(axes, entry["variants"]):
        ax.imshow(Image.open(path))
        ax.set_title(Path(path).name)
        ax.axis("off")
    plt.suptitle(f"{category}: {subject_id}")
    plt.show()

# Example: show_variants("characters", "monkey")

In [ ]:
def set_canonical(category, subject_id, variant_number):
    entry = manifest[category][subject_id]
    src = Path(entry["variants"][variant_number - 1])
    dst = src.parent / "canonical.png"
    dst.write_bytes(src.read_bytes())
    print(f"canonical set: {dst}")

# Example usage after reviewing the grid above:
# set_canonical("characters", "monkey", 2)
# set_canonical("places", "riverbank", 1)